In [45]:
import cv2
import numpy as np
import pyttsx3
import threading
import queue
import time

from tensorflow import keras
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from cvzone.HandTrackingModule import HandDetector
from collections import Counter


# ==========================================
# HAND SIGN DETECTOR
# ==========================================

class HandSignDetector:

    def __init__(
        self,
        model_path="sign_alphabet_mob_model.keras",
        class_indices_path="class_indices_mob.npy"
    ):

        print("Loading model...")

        self.model = keras.models.load_model(
            model_path,
            compile=False
        )

        print("Model loaded")


        class_indices = np.load(
            class_indices_path,
            allow_pickle=True
        ).item()


        self.idx_to_class = {
            v:k for k,v in class_indices.items()
        }


        print(self.idx_to_class)


        self.detector = HandDetector(
            maxHands=2,
            detectionCon=0.7
        )


        self.img_size = 224


        # prediction smoothing

        self.prediction_buffer = []


        # sentence

        self.sentence = ""
        self.last_letter = ""

        # voice tracking

        self.last_spoken = ""



        # ======================
        # VOICE QUEUE
        # ======================

        self.voice_queue = queue.Queue()


        self.voice_thread = threading.Thread(
            target=self.voice_worker,
            daemon=True
        )

        self.voice_thread.start()



    # ==========================================
    # VOICE WORKER
    # ==========================================

    def voice_worker(self):

        import win32com.client
    
        while True:
    
            text = self.voice_queue.get()
    
    
            try:
    
                print("Speaking:", text)
    
                speaker = win32com.client.Dispatch(
                    "SAPI.SpVoice"
                )
    
    
                # female voice
                voices = speaker.GetVoices()
    
                for i in range(voices.Count):
    
                    name = voices.Item(i).GetDescription()
    
                    if "female" in name.lower() or "zira" in name.lower():
    
                        speaker.Voice = voices.Item(i)
                        break
    
    
    
                speaker.Rate = -1     # slower
                speaker.Volume = 100  # max
    
    
                speaker.Speak(text)
    
    
    
            except Exception as e:
    
                print(
                    "Voice error:",
                    e
                )
    
    
            self.voice_queue.task_done()

    def speak(self,text):

        if text == self.last_spoken:
            return
    
    
        self.last_spoken = text
    
    
        self.voice_queue.put(text)
    
        print(
            "Voice queued:",
            text
        )
    # ==========================================
    # SENTENCE BUILDER
    # ==========================================

    def update_sentence(self, letter):

        if letter == self.last_letter:
            return
    
    
        self.sentence += letter
    
        self.last_letter = letter


    # ==========================================
    # PREPROCESS
    # ==========================================

    def preprocess_image(self,crop):


        canvas = np.zeros(
            (224,224,3),
            dtype=np.uint8
        )


        h,w = crop.shape[:2]


        if h==0 or w==0:

            return None,None



        scale=min(
            224/w,
            224/h
        )


        nw=int(w*scale)
        nh=int(h*scale)



        resized=cv2.resize(
            crop,
            (nw,nh)
        )


        x=(224-nw)//2
        y=(224-nh)//2



        canvas[
            y:y+nh,
            x:x+nw
        ] = resized



        processed = preprocess_input(
            canvas.astype(np.float32)
        )


        processed=np.expand_dims(
            processed,
            axis=0
        )


        return processed,canvas




    # ==========================================
    # PREDICT
    # ==========================================

    def predict(self,img):

        try:


            result=self.detector.findHands(
                img,
                draw=False
            )


            if isinstance(result,tuple):

                hands=result[0]
                img=result[1]

            else:

                hands=result



            if hands is None or len(hands)==0:

                self.last_letter=""
                self.last_spoken=""

                return None,0,None,None

            xs=[]
            ys=[]



            for hand in hands:

                for lm in hand["lmList"]:

                    xs.append(int(lm[0]))
                    ys.append(int(lm[1]))



            pad=60


            h,w=img.shape[:2]


            x1=max(0,min(xs)-pad)
            y1=max(0,min(ys)-pad)

            x2=min(w,max(xs)+pad)
            y2=min(h,max(ys)+pad)



            crop=img[
                y1:y2,
                x1:x2
            ]



            inp,processed=self.preprocess_image(
                crop
            )



            prediction=self.model.predict(
                inp,
                verbose=0
            )



            index=np.argmax(
                prediction[0]
            )


            confidence=float(
                prediction[0][index]
            )


            label=self.idx_to_class[index]



            # smoothing

            self.prediction_buffer.append(label)


            if len(self.prediction_buffer)>5:

                self.prediction_buffer.pop(0)



            stable,count = Counter(
                self.prediction_buffer
            ).most_common(1)[0]



            

           # only confident letters

            if count >= 4 and confidence > 0.70:

                if stable != self.last_letter:
            
                    self.sentence += stable
            
                    self.last_letter = stable
            
                    self.speak(stable)


            return (
                stable,
                confidence,
                (x1,y1,x2,y2),
                processed
            )



        except Exception as e:

            print(
                "Prediction error:",
                e
            )


            return None,0,None,None





# ==========================================
# MAIN
# ==========================================


def main():


    cap=cv2.VideoCapture(0)

    cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    1280
    )

    cap.set(
        cv2.CAP_PROP_FRAME_HEIGHT,
        720
    )


    detector=HandSignDetector()



    while True:


        success,img=cap.read()


        if not success:
            continue



        img=cv2.flip(
            img,
            1
        )



        label,conf,bbox,processed=detector.predict(
            img
        )



        if bbox:


            x1,y1,x2,y2=bbox


            cv2.rectangle(
                img,
                (x1,y1),
                (x2,y2),
                (0,255,0),
                2
            )



     # =====================
     # FULL CAMERA VIEW + SMALL OVERLAY
     # =====================

        frame_h, frame_w = img.shape[:2]
        
        
        # small top-right transparent box
        
        cv2.rectangle(
            img,
            (frame_w-320, 10),
            (frame_w-10, 170),
            (0,0,0),
            -1
        )
        
        
        
        if label:
        
            cv2.putText(
                img,
                f"{label}",
                (frame_w-290,55),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.8,
                (0,255,0),
                3
            )
        
        
            cv2.putText(
                img,
                f"{conf*100:.1f}%",
                (frame_w-290,90),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0,255,255),
                2
            )
        
        
        
        # sentence
        
        cv2.putText(
            img,
            "Text:",
            (frame_w-290,125),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255,255,0),
            2
        )
        
        
        # show last part only
        
        sentence_display = detector.sentence[-18:]
        
        
        cv2.putText(
            img,
            sentence_display,
            (frame_w-290,155),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255,255,255),
            2
        )
        cv2.imshow(
            "Sign Language To Text",
            img
        )



        # model input

        if processed is not None:

            cv2.imshow(
                "Model Input",
                processed
            )



        key=cv2.waitKey(1)



        # space

        if key==ord(' '):

            detector.sentence += " "

            detector.last_letter=""



        # clear

        if key==ord('c'):

            detector.sentence=""



        # backspace

        if key==8:

            detector.sentence = detector.sentence[:-1]

       # speak complete sentence
        if key == 13:   # Enter key

           if detector.sentence.strip():

              detector.speak(
              detector.sentence
              )


        if key==ord('q'):

            break



    cap.release()

    cv2.destroyAllWindows()



if __name__=="__main__":

    main()

Loading model...
Model loaded
{0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'j', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'q', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'x', 24: 'y', 25: 'z'}
Voice queued:Speaking: x
 x
Voice queued:Speaking: w
 w
Voice queued:Speaking: w
 w
Voice queued:Speaking: h
 h
Voice queued:Speaking: o
 o
Voice queued:Speaking: q
 q
Voice queued: i
Speaking: i
Voice queued:Speaking: i
 i
Voice queued:Speaking: q
 q
Voice queued:Speaking: i
 i
Voice queued:Speaking: o
 o
Voice queued: a
Speaking: a
Voice queued:Speaking: m
 m
Voice queued:Speaking: m
 m
Voice queued:Speaking: a
 a
Voice queued:Speaking: a
 a
Voice queued:Speaking: m
 m
Voice queued:Speaking: m
 m
Voice queued:Speaking: r
 r
Voice queued:Speaking: r
 r
Voice queued:Speaking: b
 b
Voice queued:Speaking: w
 w
Voice queued:Speaking: m
 m
Voice queued:Speaking: m
 m
Voice queued:Speaking: a
 a
Voice queued:Speaking: a
 a
Voice queued: w
S